# 00 — S01 environment and release-boundary audit

**Question:** Can the public package be imported and audited without touching protected or participant-level material?

**Permitted inputs:** repository metadata, public source code, `config/protocol.example.yaml`, and `data/README.md`.

**Output:** `notebooks/artifacts/00_preflight_report.json`. This notebook does not use E01 aggregate values as analytical evidence or reproduce E01 findings. The release validator checks only scope and protected-access metadata.

In [1]:
from __future__ import annotations

import json
import platform
import subprocess
import sys
from importlib.metadata import version
from pathlib import Path

from neurofusion_qa.provenance import canonical_json_sha256, sha256_file

START = Path.cwd().resolve()
ROOT = next(
    candidate for candidate in (START, *START.parents)
    if (candidate / 'pyproject.toml').is_file()
    and (candidate / 'src' / 'neurofusion_qa').is_dir()
)
ARTIFACT_DIR = ROOT / 'notebooks' / 'artifacts'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Repository: {ROOT.name}')
print(f'Python: {platform.python_version()}')

Repository: NeuroFusion-QA
Python: 3.12.14


## Record the executable environment

Versions are evidence about this run, not a claim that every platform has already been validated.

In [2]:
packages = {name: version(name) for name in ('neurofusion-qa', 'numpy', 'scipy', 'scikit-learn')}
packages

{'neurofusion-qa': '0.1.0',
 'numpy': '2.5.3',
 'scipy': '1.18.1',
 'scikit-learn': '1.9.1'}

## Execute the fail-closed public-release validator

A pass establishes the documented release boundary only. It does not establish scientific QC or clinical validity.

In [3]:
validation = subprocess.run(
    [sys.executable, str(ROOT / 'scripts' / 'validate_release.py')],
    cwd=ROOT,
    check=False,
    capture_output=True,
    text=True,
)
assert validation.returncode == 0, validation.stdout + validation.stderr
print(validation.stdout.strip())

Release validation passed: aggregate-only, protected access count 0, no prohibited artifacts.


## Confirm the tutorial input surface

The public `data/` directory must contain documentation only at this stage. Synthetic arrays are generated in later notebooks and are summarized as JSON rather than serialized as participant-like files.

In [4]:
permitted_inputs = sorted(
    path.relative_to(ROOT).as_posix()
    for path in (ROOT / 'data').rglob('*')
    if path.is_file()
)
assert permitted_inputs == ['data/README.md'], permitted_inputs
assert not any((ROOT / 'data').rglob('*.npy'))
assert not any((ROOT / 'data').rglob('*.npz'))
permitted_inputs

['data/README.md']

## Write and independently verify the machine-readable preflight record

In [5]:
report = {
    'schema_version': '1.0',
    'notebook_id': '00_S01_Environment_Boundary_Audit',
    'boundary': {
        'input_scope': 'public_metadata_and_documentation_only',
        'e01_participant_level_access': False,
        'e01_results_consumed': False,
        'release_boundary_metadata_checked': True,
        'release_validator': validation.stdout.strip(),
    },
    'environment': {
        'python': platform.python_version(),
        'implementation': platform.python_implementation(),
        'platform_system': platform.system(),
        'platform_machine': platform.machine(),
        'packages': packages,
    },
    'protocol': {
        'path': 'config/protocol.example.yaml',
        'sha256': sha256_file(ROOT / 'config' / 'protocol.example.yaml'),
    },
    'permitted_input_files': permitted_inputs,
    'seed_policy': {
        'synthetic_fixture_seed': 20260918,
        'model_seed': 1201,
        'bootstrap_seed': 20260816,
    },
}
report['canonical_content_sha256'] = canonical_json_sha256(report)
artifact_path = ARTIFACT_DIR / '00_preflight_report.json'
artifact_path.write_text(json.dumps(report, indent=2, sort_keys=True) + '\n', encoding='utf-8')
reloaded = json.loads(artifact_path.read_text(encoding='utf-8'))
stored_hash = reloaded.pop('canonical_content_sha256')
assert canonical_json_sha256(reloaded) == stored_hash
assert reloaded['boundary']['e01_participant_level_access'] is False
assert reloaded['boundary']['e01_results_consumed'] is False
{'artifact': artifact_path.relative_to(ROOT).as_posix(), 'sha256': sha256_file(artifact_path)}

{'artifact': 'notebooks/artifacts/00_preflight_report.json',
 'sha256': 'e3a32eec9d8a7ab2ba37f5d91d5326139eccbb7798c58a86b7d6115aebbaa29f'}

## Evidence-bound figure exports

Regenerate relevant figures from this execution's evidence. Every family has editable SVG/PDF and 1000 dpi Times New Roman PNG exports, a caption, and source hashes. [Full gallery](figures/README.md). Synthetic software evidence only.

In [6]:
import sys
from IPython.display import Markdown, display
if str(ROOT / 'scripts') not in sys.path:
    sys.path.insert(0, str(ROOT / 'scripts'))
from s01_evidence_figures import generate
figure_records = generate('00')
for item in figure_records:
    display(Markdown(f"### {item['id']}\n\n![{item['id']}](figures/previews/{item['id']}.png)\n\n{item['caption']}"))
{'figure_sets': len(figure_records), 'export_formats': ['png', 'svg', 'pdf']}

### F00_environment

![F00_environment](figures/previews/F00_environment.png)

Environment and boundary audit for this execution, not a portability certification.

{'figure_sets': 1, 'export_formats': ['png', 'svg', 'pdf']}

Exported F00_environment: Times New Roman, 1000 dpi
